# Compare Performance

In [ ]:
from glow.compare_analyses import load_update_all, get_path_result

# prints all experiments (useful to pick a folder)
path_result = get_path_result()
sorted(path_result.glob('exp_*'))

In [ ]:
# load data
df = load_update_all()

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# -------------------- parameters --------------------
x_param = 'hotel_tr'
metrics = ['f1', 'sens', 'spec']
alpha = .1

# colormap to use; colors assigned in sorted label order: tab10(0), tab10(1), ...
tab10 = plt.get_cmap('tab10')

# bottom-row comparison: set the two labels to compare
# if either label is not present in the data, the bottom row will be omitted
diff_label_a = 'HGLM'
diff_label_b = 'TFCE'
# ---------------------------------------------------

# ensure numeric x + metrics (prevents lexicographic sorts)
df2 = df.copy()
df2[x_param] = pd.to_numeric(df2[x_param], errors='coerce')
for m in metrics:
    df2[m] = pd.to_numeric(df2[m], errors='coerce')
df2 = df2.dropna(subset=[x_param])

# aggregate to unique (label, seed, x_param)
df_agg = (df2.groupby(['label', 'seed', x_param], as_index=False)[metrics]
            .mean())

# build color map by sorted labels → tab10(0), tab10(1), ...
labels_sorted = sorted(df_agg['label'].unique().tolist())
num_colors = getattr(tab10, 'N', len(getattr(tab10, 'colors', [])) or 10)
color_map = {lab: tab10(i % num_colors) for i, lab in enumerate(labels_sorted)}

# determine if we can plot the bottom diff row
labels_present = set(labels_sorted)
plot_bottom = (diff_label_a in labels_present) and (diff_label_b in labels_present)

nrows = 2 if plot_bottom else 1
fig, axes = plt.subplots(nrows, 3, figsize=(14, 5.5 if plot_bottom else 3.0), sharex='col')
if nrows == 1:
    axes = np.atleast_2d(axes)  # normalize indexing so axes[0, j] works

for j, metric in enumerate(metrics):
    ax_top = axes[0, j]

    # top: per-seed lines + bold mean
    for label, sub in df_agg.groupby('label'):
        color = color_map[label]

        for seed, g in sub.groupby('seed', sort=False):
            g = g.sort_values(x_param)
            ax_top.plot(g[x_param], g[metric], lw=.5, alpha=alpha, color=color)

        mean_curve = (sub.groupby(x_param, as_index=False)[metric]
                        .mean()
                        .sort_values(x_param))
        ax_top.plot(mean_curve[x_param], mean_curve[metric],
                    lw=3, color=color, label=label)

    if j == 0:
        ax_top.legend(frameon=False)
    ax_top.set_title(metric)
    ax_top.grid(True, alpha=alpha, linewidth=1.2)

    # top x-label if only one row
    if not plot_bottom:
        ax_top.set_xlabel(x_param)

    # bottom: specified diff (only if both labels are present)
    if plot_bottom:
        ax_bot = axes[1, j]
        lab_a, lab_b = diff_label_a, diff_label_b

        pivot = (df_agg.pivot_table(index=['seed', x_param],
                                    columns='label',
                                    values=metric)
                       .reset_index())

        missing_cols = [c for c in (lab_a, lab_b) if c not in pivot.columns]
        if not missing_cols:
            pivot = pivot.dropna(subset=[lab_a, lab_b])
            if not pivot.empty:
                pivot['diff'] = pivot[lab_a] - pivot[lab_b]

                for seed, g in pivot.groupby('seed', sort=False):
                    g = g.sort_values(x_param)
                    ax_bot.plot(g[x_param], g['diff'], lw=.5, alpha=alpha, color='black')

                diff_mean = (pivot.groupby(x_param, as_index=False)['diff']
                                   .mean()
                                   .sort_values(x_param))
                ax_bot.plot(diff_mean[x_param], diff_mean['diff'], lw=3, color='black')
                ax_bot.set_ylabel(f'{lab_a} - {lab_b}')
                ax_bot.axhline(0, lw=.5, color='black', alpha=alpha)
                ax_bot.grid(True, alpha=alpha, linewidth=1.2)
            else:
                ax_bot.axis('off')
        else:
            ax_bot.axis('off')

        # always set x-label for bottom row
        ax_bot.set_xlabel(x_param)

xmin = df_agg[x_param].min()
if pd.notnull(xmin) and xmin > 0:
    for r in range(nrows):
        for ax in axes[r]:
            ax.set_xscale('log')

axes[0, 0].set_ylabel('score')

plt.tight_layout()
plt.show()


# Compare Computation Time

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

sns.swarmplot(data=df, x='time_sec', hue='label', size=3)
plt.suptitle('Time per experiment')
plt.xlabel('Time (seconds)')
plt.gcf().set_size_inches(10, 5)

# Find "worst case" difference

In [ ]:
import pandas as pd

# Merge HGLM with corresponding TFCE rows on hotel_tr, rough, seed
df_hglm = df[df['Analysis'] == 'AnalysisHGLM']
df_tfce = df[df['Analysis'] == 'AnalysisTFCE']

merged = pd.merge(
    df_hglm,
    df_tfce,
    on=['hotel_tr', 'seed'],
    suffixes=('_HGLM', '_TFCE')
)

# Calculate difference: TFCE.spec - HGLM.spec
merged['spec_diff'] = merged['spec_TFCE'] - merged['spec_HGLM']

# Find the row with the largest difference
max_diff_row = merged.loc[merged['spec_diff'].idxmax()]

# Extract HGLM uuid and original row index
uuid_hglm = max_diff_row['uuid_HGLM']

uuid_hglm

In [ ]:
max_diff_row